In [ ]:
# Optional: Install required dependencies if not already present in your Kaggle environment
!pip install -q tensorly datasets scikit-learn


# Experiment 12: Model-Wide Independent Sublayer DBSCAN Tucker Compression (Kaggle Version)

**Target Model**: `google/gemma-3-1b-it` (All 26 Transformer Decoder Layers: `model.layers[0...25]`)
**Target Submodules**: All 3 MLP Projections (`gate_proj`, `up_proj`, and `down_proj` -> 78 total matrices)
**Evaluation Tasks**:
1. **GLUE MNLI Classification Benchmark**: 1,000 validation samples (`validation_matched`)
2. **Open-Ended Generative Validation**: Qualitative text generation (Chocolate Cake Recipe query)

### Finding the True Sweet Spot on 1k Samples:
- **Why 77% Recon Error Collapsed**: Early single-layer experiments (Layer 0) tolerated ~77% reconstruction error (`[4, 180, 600]`). However, when compounded through all 26 decoder layers in sequence, ~77% error per matrix completely scrambled residual stream activations, dropping downstream accuracy to ~33% (random chance) and producing incoherent token generation.
- **The 1k-Sample Sweep**: To find the true Pareto-optimal sweet spot where parameter reduction is maximized while downstream reasoning is preserved, this notebook sweeps 5 distinct fidelity tiers on **1,000 evaluation samples**:
  1. **Tier 1: Ultra-High Fidelity (`[5, 280, 850]`)**: Target ~58% recon error (~37.7M parameters eliminated / 6.1% of MLP).
  2. **Tier 2: High Fidelity (`[5, 240, 750]`)**: Target ~64% recon error (~70.6M parameters eliminated / 11.4% of MLP).
  3. **Tier 3: Balanced Tier (`[4, 250, 800]`)**: Target ~70% recon error (~73.6M parameters eliminated / 11.8% of MLP).
  4. **Tier 4: Moderate-73 (`[4, 220, 700]`)**: Target ~73% recon error (~97.8M parameters eliminated / 15.8% of MLP).
  5. **Tier 5: Legacy Reference (`[4, 180, 600]`)**: Target ~77% recon error (~122.4M parameters eliminated / 19.7% of MLP) to document the collapse threshold.

### Key Technical Highlights:
1. **GPU-Accelerated Tucker Adam GD**: Factorization runs directly on GPU (`sub_dev = mod_ref.weight.device`) for ~20x-50x speedup.
2. **Zero RAM Bloat**: Forward hooks pool token activations on-the-fly (`squeeze(0).mean(dim=0)`).
3. **Safe Multi-GPU Live Injection**: Submodule-level device handling (`mod_ref.weight.device`) with `torch.cuda.empty_cache()`.


In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")


In [ ]:
# =====================================================================
# STEP 1: Environment Setup, Time Logging & Standard Imports
# =====================================================================
import os
import sys
import time
from pathlib import Path
import math
import json
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer
import tensorly as tl
from tensorly.decomposition import tucker
from tensorly.tucker_tensor import tucker_to_tensor
from datasets import load_dataset
from tqdm import tqdm
from sklearn.cluster import DBSCAN
from sklearn.metrics import accuracy_score
from IPython import get_ipython

# Set TensorLy PyTorch backend
tl.set_backend("pytorch")

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Minimal Time Logging
GLOBAL_NOTEBOOK_START_TIME = time.time()
NOTEBOOK_TIMINGS = []
_current_cell_start = None

ip = get_ipython()
if ip is not None:
    def _pre_cell_hook(info):
        global _current_cell_start
        _current_cell_start = time.time()

    def _post_cell_hook(result):
        global _current_cell_start
        if _current_cell_start is not None:
            elapsed = time.time() - _current_cell_start
            cumulative = time.time() - GLOBAL_NOTEBOOK_START_TIME
            cell_id = result.execution_count or len(NOTEBOOK_TIMINGS) + 1

            timing_entry = {
                "cell_id": cell_id,
                "time": round(elapsed, 3),
                "cummulative_time": round(cumulative, 3),
            }
            NOTEBOOK_TIMINGS.append(timing_entry)

            print(f"time: {elapsed:.2f}s")
            print(f"cummulative_time: {cumulative:.2f}s")

    ip.events.register("pre_run_cell", _pre_cell_hook)
    ip.events.register("post_run_cell", _post_cell_hook)

print(f"PyTorch Version: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active Device: {torch.cuda.get_device_name(0)}")


In [ ]:
# =====================================================================
# STEP 2: Model & Dataset Loading (Pure Hugging Face - No Custom Module)
# =====================================================================
import huggingface_hub

MODEL_ID = "google/gemma-3-1b-it"
NUM_LAYERS = 26
HIDDEN_DIM = 1152
INTERMEDIATE_DIM = 6912
NUM_EVAL_SAMPLES = 1000

# Hugging Face Authentication for Gated Gemma-3 Model
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        hf_token = user_secrets.get_secret("HF_TOKEN")
    except Exception:
        hf_token = None

if hf_token:
    huggingface_hub.login(token=hf_token)
    print("Authenticated with Hugging Face via token.")
else:
    print("Warning: No HF_TOKEN found in Kaggle Secrets or environment.")

print(f"Loading model: {MODEL_ID} with device_map='auto' in FP32...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    token=hf_token,
)
model.eval()

# Cache original pristine weights on CPU across all 26 layers for calibration & rollback
W_orig_all = {}
for l in range(NUM_LAYERS):
    layer_mlp = model.model.layers[l].mlp
    W_orig_all[l] = {
        "gate_proj": layer_mlp.gate_proj.weight.data.clone().cpu(),
        "up_proj":   layer_mlp.up_proj.weight.data.clone().cpu(),
        "down_proj": layer_mlp.down_proj.weight.data.clone().cpu(),
    }

print(f"Cached pristine weights on CPU for all {NUM_LAYERS} layers (78 projection matrices).")

# Load GLUE MNLI Validation Benchmark
ds = load_dataset("nyu-mll/glue", "mnli", split="validation_matched")
eval_data = ds.select(range(NUM_EVAL_SAMPLES))

label_names = ["entailment", "neutral", "contradiction"]
label_token_ids = [tokenizer.encode(" " + name, add_special_tokens=False)[0] for name in label_names]

RECIPE_PROMPT = (
    "<start_of_turn>user\n"
    "What is the best recipe to make a chocolate cake?<end_of_turn>\n"
    "<start_of_turn>model\n"
)

def evaluate_mnli(model, desc="Evaluating"):
    predictions, ground_truth = [], []
    model.eval()
    with torch.no_grad():
        for sample in tqdm(eval_data, desc=desc):
            prompt = (
                f"<start_of_turn>user\n"
                f"Premise: {sample['premise']}\n"
                f"Hypothesis: {sample['hypothesis']}\n"
                f"Determine if the relationship between the 'Premise' and 'Hypothesis' is 'entailment', 'neutral' or 'contradiction.'\n"
                f"Answer with one word\n"
                f"<start_of_turn>model\n"
            )
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
            outputs = model(**inputs, logits_to_keep=1)
            candidate_logits = outputs.logits[0, -1, :][label_token_ids]
            predictions.append(torch.argmax(candidate_logits).item())
            ground_truth.append(sample["label"])
    acc = accuracy_score(ground_truth, predictions)
    return acc

def generate_recipe(model, max_new_tokens=256):
    model.eval()
    inputs = tokenizer(RECIPE_PROMPT, return_tensors="pt").to(model.device)
    with torch.no_grad():
        tokens = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )
    recipe_text = tokenizer.decode(tokens[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return recipe_text

print(f"Setup evaluation suite: {NUM_EVAL_SAMPLES} GLUE MNLI samples + Recipe generation test.")


In [ ]:
# =====================================================================
# STEP 3: Model-Wide Tri-Hook Profiling & Pristine Baseline Accuracy
# =====================================================================
# Memory-optimized streaming storage: pool sequence tokens immediately to prevent RAM bloat
layer_acts = {
    l: {"gate": [], "up": [], "down": []} for l in range(NUM_LAYERS)
}

hooks = []
for l in range(NUM_LAYERS):
    lmod = model.model.layers[l].mlp

    def make_gate_hook(layer_idx):
        return lambda m, inp, out: layer_acts[layer_idx]["gate"].append(
            out.detach().squeeze(0).mean(dim=0).float().cpu().numpy()
        )

    def make_up_hook(layer_idx):
        return lambda m, inp, out: layer_acts[layer_idx]["up"].append(
            out.detach().squeeze(0).mean(dim=0).float().cpu().numpy()
        )

    def make_down_hook(layer_idx):
        return lambda m, inp, out: layer_acts[layer_idx]["down"].append(
            inp[0].detach().squeeze(0).mean(dim=0).float().cpu().numpy()
        )

    hooks.append(lmod.act_fn.register_forward_hook(make_gate_hook(l)))
    hooks.append(lmod.up_proj.register_forward_hook(make_up_hook(l)))
    hooks.append(lmod.down_proj.register_forward_hook(make_down_hook(l)))

print(f"Running baseline profiling pass across all {NUM_LAYERS} layers ({NUM_EVAL_SAMPLES} samples)...\n")
baseline_accuracy = evaluate_mnli(model, desc="Simultaneous Tri-Hook Profiling & MNLI Baseline")

for h in hooks:
    h.remove()

acts_all = {
    l: {
        "gate": np.stack(layer_acts[l]["gate"]),
        "up":   np.stack(layer_acts[l]["up"]),
        "down": np.stack(layer_acts[l]["down"]),
    }
    for l in range(NUM_LAYERS)
}

# Free intermediate raw lists
del layer_acts

print(f"\nUncompressed Pristine Baseline Accuracy: {baseline_accuracy * 100:.2f}%")
print(f"Captured tri-hook activation matrices for all {NUM_LAYERS} layers (78 submodules).")

print("\nGenerating Baseline Recipe:")
pristine_recipe = generate_recipe(model)
print(f"{'='*95}\n{pristine_recipe}\n{'='*95}")


In [ ]:
# =====================================================================
# STEP 4: Independent DBSCAN Pre-Clustering for All 78 Submodules
# =====================================================================
CHUNK_SIZE = 400
NUM_CHUNKS = 6

def cluster_submodule(acts_matrix, weight_tensor, is_col=False):
    v = np.mean(acts_matrix, axis=0)
    std_v = np.std(v)
    eps = max(0.04, float(std_v * 0.18))

    db = DBSCAN(eps=eps, min_samples=30, metric="euclidean")
    labels = db.fit_predict(v.reshape(-1, 1))

    max_mags = np.max(np.abs(acts_matrix), axis=0)
    variances = np.var(acts_matrix, axis=0)
    super_mask = (labels == -1) | (max_mags > 3.0) | (variances >= np.quantile(variances, 0.99))
    super_indices = np.where(super_mask)[0]

    unique_labels = [lab for lab in np.unique(labels) if lab != -1]

    chunk_list = []
    for lab in unique_labels:
        c_idx = np.where((labels == lab) & (~super_mask))[0]
        if len(c_idx) == 0:
            continue
        sorted_idx = c_idx[np.argsort(v[c_idx])]
        num_full = len(sorted_idx) // CHUNK_SIZE
        for ci in range(num_full):
            chunk_list.append(sorted_idx[ci * CHUNK_SIZE : (ci + 1) * CHUNK_SIZE])
            if len(chunk_list) >= NUM_CHUNKS:
                break
        if len(chunk_list) >= NUM_CHUNKS:
            break

    if len(chunk_list) < NUM_CHUNKS:
        assigned = set(np.concatenate(chunk_list) if chunk_list else [])
        avail = [i for i in range(len(v)) if i not in assigned and not super_mask[i]]
        needed = NUM_CHUNKS - len(chunk_list)
        for ci in range(needed):
            if len(avail) >= CHUNK_SIZE:
                chunk_list.append(np.array(avail[:CHUNK_SIZE]))
                avail = avail[CHUNK_SIZE:]

    active_coords = np.concatenate(chunk_list)

    if is_col:
        T = torch.stack([weight_tensor[:, c].T.float().cpu() for c in chunk_list], dim=0)
    else:
        T = torch.stack([weight_tensor[c, :].float().cpu() for c in chunk_list], dim=0)

    return {
        "tensor": T,
        "chunk_list": chunk_list,
        "active_coords": active_coords,
        "super_indices": super_indices,
        "is_col": is_col,
    }

print("Running independent DBSCAN clustering across all 26 layers...")
layer_submodule_data = {}

for l in range(NUM_LAYERS):
    layer_submodule_data[l] = {
        "gate_proj": cluster_submodule(acts_all[l]["gate"], W_orig_all[l]["gate_proj"], is_col=False),
        "up_proj":   cluster_submodule(acts_all[l]["up"],   W_orig_all[l]["up_proj"],   is_col=False),
        "down_proj": cluster_submodule(acts_all[l]["down"], W_orig_all[l]["down_proj"], is_col=True),
    }

print(f"Pre-clustering complete for all {NUM_LAYERS} layers (78 independent clusterings).")


In [ ]:
# =====================================================================
# STEP 5: GPU-Accelerated Tucker GD Optimizer & Multi-Tier Sweep
# =====================================================================
def optimize_tucker_gd(T, ranks, num_steps=35, lr=1e-3, device=None):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    safe_ranks = [
        min(ranks[0], T.shape[0]),
        min(ranks[1], T.shape[1]),
        min(ranks[2], T.shape[2]),
    ]
    T_target = T.to(device)
    core_init, factors_init = tucker(T_target, rank=safe_ranks, init='svd')
    core_param = torch.nn.Parameter(core_init.clone())
    factors_param = [torch.nn.Parameter(f.clone()) for f in factors_init]
    optimizer = torch.optim.Adam([core_param] + factors_param, lr=lr)

    for step in range(num_steps):
        optimizer.zero_grad()
        T_recon = tucker_to_tensor((core_param, factors_param))
        loss = torch.norm(T_target - T_recon) ** 2
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        T_recon_final = tucker_to_tensor((core_param, factors_param))
        final_err = (torch.norm(T_target - T_recon_final) / torch.norm(T_target)).item()

    return core_param.detach().cpu(), [f.detach().cpu() for f in factors_param], T_recon_final, final_err

eval_tiers = [
    {
        "name": "Tier 1: Ultra-High Fidelity ([5, 280, 850])",
        "target_error": "~58%",
        "ranks": {
            "gate_proj": [5, 280, 850],
            "up_proj":   [5, 280, 850],
            "down_proj": [5, 280, 850],
        }
    },
    {
        "name": "Tier 2: High Fidelity ([5, 240, 750])",
        "target_error": "~64%",
        "ranks": {
            "gate_proj": [5, 240, 750],
            "up_proj":   [5, 240, 750],
            "down_proj": [5, 240, 750],
        }
    },
    {
        "name": "Tier 3: Balanced Tier ([4, 250, 800])",
        "target_error": "~70%",
        "ranks": {
            "gate_proj": [4, 250, 800],
            "up_proj":   [4, 250, 800],
            "down_proj": [4, 250, 800],
        }
    },
    {
        "name": "Tier 4: Moderate-73 ([4, 220, 700])",
        "target_error": "~73%",
        "ranks": {
            "gate_proj": [4, 220, 700],
            "up_proj":   [4, 220, 700],
            "down_proj": [4, 220, 700],
        }
    },
    {
        "name": "Tier 5: Legacy 77% Tier ([4, 180, 600])",
        "target_error": "~77%",
        "ranks": {
            "gate_proj": [4, 180, 600],
            "up_proj":   [4, 180, 600],
            "down_proj": [4, 180, 600],
        }
    },
]

print(f"Defined GPU Tucker GD optimizer and 5-tier sweep ({len(eval_tiers)} tiers) across 1k samples.")


In [ ]:
# =====================================================================
# STEP 6: Execute Multi-Tier Compression Sweeps & Downstream Evaluation
# =====================================================================
tier_benchmarks = []
tier_recipes = {}

for tier in eval_tiers:
    tier_name = tier["name"]
    tier_ranks = tier["ranks"]

    print(f"\n{'='*95}")
    print(f"Running Full-Model Evaluation for: {tier_name}")
    print(f"{'='*95}")

    tier_gate_errs, tier_up_errs, tier_down_errs = [], [], []
    total_params_saved = 0

    # 1. Factorize and inject across all 26 layers on GPU
    for l in range(NUM_LAYERS):
        lmod = model.model.layers[l].mlp
        sdata_layer = layer_submodule_data[l]

        for sub_name in ["gate_proj", "up_proj", "down_proj"]:
            sdata = sdata_layer[sub_name]
            ranks = tier_ranks[sub_name]
            mod_ref = getattr(lmod, sub_name)
            sub_dev = mod_ref.weight.device

            cg, fg, T_recon, err = optimize_tucker_gd(
                sdata["tensor"], ranks=ranks, num_steps=35, lr=1e-3, device=sub_dev
            )

            if sub_name == "gate_proj": tier_gate_errs.append(err)
            elif sub_name == "up_proj":  tier_up_errs.append(err)
            elif sub_name == "down_proj": tier_down_errs.append(err)

            orig_p = sdata["tensor"].numel()
            comp_p = cg.numel() + sum(f.numel() for f in fg)
            total_params_saved += (orig_p - comp_p)

            # Live injection on module device
            orig_w = W_orig_all[l][sub_name]
            mod_ref.weight.data = orig_w.clone().to(sub_dev)

            if sdata["is_col"]:
                for k, c in enumerate(sdata["chunk_list"]):
                    mod_ref.weight.data[:, c] = T_recon[k].T.to(device=sub_dev, dtype=mod_ref.weight.dtype)
                if len(sdata["super_indices"]) > 0:
                    mod_ref.weight.data[:, sdata["super_indices"]] = orig_w[:, sdata["super_indices"]].to(sub_dev)
            else:
                for k, c in enumerate(sdata["chunk_list"]):
                    mod_ref.weight.data[c, :] = T_recon[k].to(device=sub_dev, dtype=mod_ref.weight.dtype)
                if len(sdata["super_indices"]) > 0:
                    mod_ref.weight.data[sdata["super_indices"], :] = orig_w[sdata["super_indices"], :].to(sub_dev)

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    mean_gate_err = float(np.mean(tier_gate_errs) * 100)
    mean_up_err   = float(np.mean(tier_up_errs) * 100)
    mean_down_err = float(np.mean(tier_down_errs) * 100)

    print(f"Layer Factorization Complete. Mean Recon Errors: gate={mean_gate_err:.1f}%, up={mean_up_err:.1f}%, down={mean_down_err:.1f}%")
    print(f"Total Parameters Eliminated: {total_params_saved:,}")

    # 2. Evaluate on GLUE MNLI
    acc = evaluate_mnli(model, desc=f"Evaluating {tier_name}")
    delta = acc - baseline_accuracy

    print(f"\nResult for {tier_name}:")
    print(f"  Downstream Accuracy: {acc * 100:.2f}% (Δ vs Baseline: {delta * 100:+.2f}%)")
    print(f"  Total Params Cut:    {total_params_saved:,}")

    # 3. Qualitative Generation
    recipe_gen = generate_recipe(model)
    tier_recipes[tier_name] = recipe_gen
    print(f"\nSample Generation for {tier_name}:\n{'-'*60}\n{recipe_gen}\n{'-'*60}")

    tier_benchmarks.append({
        "Variant": tier_name,
        "Target_Error": tier.get("target_error", ""),
        "Mean_Gate_Err": round(mean_gate_err, 2),
        "Mean_Up_Err": round(mean_up_err, 2),
        "Mean_Down_Err": round(mean_down_err, 2),
        "Params_Eliminated": total_params_saved,
        "Accuracy": round(acc * 100, 2),
        "Delta": round(delta * 100, 2),
    })

# Restore pristine model weights across all 26 layers
for l in range(NUM_LAYERS):
    for sub_name in ["gate_proj", "up_proj", "down_proj"]:
        sub_mod = getattr(model.model.layers[l].mlp, sub_name)
        sub_mod.weight.data = W_orig_all[l][sub_name].clone().to(sub_mod.weight.device)

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\nRestored all 26 layers to pristine weights.")


In [ ]:
# =====================================================================
# STEP 7: Benchmark Summary & JSON Artifact Export
# =====================================================================
total_mlp_params_orig = NUM_LAYERS * 3 * (HIDDEN_DIM * INTERMEDIATE_DIM)  # 621,084,672

print("=" * 125)
print(f"{'Variant':<48} | {'Target':<7} | {'Gate Err':<9} | {'Up Err':<8} | {'Down Err':<9} | {'Params Cut':<11} | {'Accuracy':<9} | {'Delta':<8}")
print("=" * 125)
print(f"{'Baseline (Uncompressed)':<48} | {'0.0%':<7} | {'0.00%':<9} | {'0.00%':<8} | {'0.00%':<9} | {'0':<11} | {baseline_accuracy*100:>7.2f}% | {'+0.00%':<8}")

for r in tier_benchmarks:
    tgt = r.get('Target_Error', '')
    print(f"{r['Variant']:<48} | {tgt:<7} | {r['Mean_Gate_Err']:>6.2f}% | {r['Mean_Up_Err']:>5.2f}% | {r['Mean_Down_Err']:>6.2f}% | {r['Params_Eliminated']:>10,} | {r['Accuracy']:>7.2f}% | {r['Delta']:>+6.2f}%")

print("=" * 125)

artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(parents=True, exist_ok=True)
results_file = artifacts_dir / "12_all_layers_independent_sublayer_results.json"

output_payload = {
    "experiment": "12_all_layers_independent_sublayer_dbscan",
    "target_model": MODEL_ID,
    "num_layers": NUM_LAYERS,
    "num_projections": NUM_LAYERS * 3,
    "num_eval_samples": NUM_EVAL_SAMPLES,
    "total_mlp_params_orig": total_mlp_params_orig,
    "baseline_accuracy": round(baseline_accuracy * 100, 2),
    "tiers": tier_benchmarks,
    "timings": NOTEBOOK_TIMINGS,
}

with open(results_file, "w") as f:
    json.dump(output_payload, f, indent=2)

print(f"Saved 1k-sample benchmark results to {results_file}")
